# A small pilot for the distillation settings

Notebook 28's kd run distils from B1 with a weight of 1.0, a temperature of 2 and a learning rate of 2e-4,
settings that were not chosen on a pilot. This pilot compares them, on the 16% experiment subset only,
with A3's own losses (no distillation) and with one other distillation setting: each starts from A3's
`best.pt`, trains 2 epochs on the subset's training photos cut by the M4 front end, and is scored on every
validation photo. The pick is by the G3 slice score (ours minus B1, top-1 and macro-F1 over B1's 62
species, photos from 2022) and by validation macro-F1 over all species. Calibration photos are not used.
B1 runs on the subset's training photos of its 62 species on the second GPU while the first builds the
front end. About 1.5 hours on two T4 GPUs.

In [ ]:
import time
STARTED = time.time()

%pip install -q timm==1.0.31 albumentations==2.0.8 scikit-learn==1.9.1
%pip install -q ultralytics==8.4.202 segmentation-models-pytorch==0.6.0
import hashlib, json, os, shutil, subprocess, sys, traceback, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from types import SimpleNamespace

os.environ.update(NO_ALBUMENTATIONS_UPDATE="1", YOLO_VERBOSE="False",
                  HF_HUB_DISABLE_PROGRESS_BARS="1", TQDM_DISABLE="1")
warnings.filterwarnings("ignore")

import kagglehub, torch, yaml
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    REPO = next(p for p in (Path("/kaggle/working/repo"), Path("/kaggle/input/serpentguard-code"))
                if (p / "src").exists())
else:
    REPO = Path("..").resolve()
sys.path.insert(0, str(REPO / "src"))
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("work_m5")
SCRATCH = Path("/kaggle/tmp/m5_pilot") if ON_KAGGLE else WORK / "tmp"

from serpentguard import data, front_end, robustness
from serpentguard.decide import decision_report, load_outputs
from serpentguard.metrics import (BOOTSTRAP_RESAMPLES, BOOTSTRAP_SEED, assert_no_test_paths,
                                  class_tables, paired_difference_ci, photographer_groups, softmax,
                                  species_report, write_json)
from serpentguard.train import (CALIBRATION_OUTPUTS, UNKNOWN_OUTPUTS, VAL_OUTPUTS, load_checkpoint,
                                predict, save_outputs, train)

ACCOUNT = "jayesh"     # the Kaggle account this session runs on
VARIANT = "kd"   # finetune, or kd (adds distillation from B1)
EPOCHS = 5             # fine-tuning epochs, each evaluated
LR = 2e-4              # A3 trained at 9.5e-4; the encoder learns at a tenth of this
KD_WEIGHT, KD_TEMPERATURE = 1.0, 2.0
SMOKE_PHOTOS, SMOKE_EPOCHS = 300, 1
TIME_LIMIT_HOURS = 11
TEACHER_WAIT_HOURS = 2.5  # the session time by which the full pass starts with whatever B1 finished
assert VARIANT in ("finetune", "kd"), VARIANT
RUN_ID = {"finetune": "M5_FT", "kd": "M5_KD"}[VARIANT]
PASSED = set()
SMOKE_PASSED = FULL_PASSED = False
for folder in (WORK, SCRATCH):
    folder.mkdir(parents=True, exist_ok=True)
print(f"{torch.cuda.device_count()} GPU(s), account {ACCOUNT}, variant {VARIANT}; "
      f"results go to {WORK}")

## 1. The inputs

The cleaned photos, the experiment subset and the SAM masks are shared with the team. The M4 front end
(the detector and U-Net of the gate notebook, `puneetbajajiitm/sg-m4-gate`) and A3's `best.pt`
(`jayeshbansal497/sg-m4-full-a3`) are copied into one private dataset; M3's front-end file is read only
to rebuild A3's scenery bank exactly; B1's saved validation predictions and M3's final run come from
notebook 21's private copies. Every file is checked by its SHA-256.

In [ ]:
ACCOUNTS = {"puneet": "puneetbajajiitm", "jayesh": "jayeshbansal497", "vikas": "vikas25s"}
SHARED = "puneetbajajiitm"
CHECKSUMS = {
    "detector": "29a1ad1c1b9b561c35ef4ffa1916998837c3c78525c21b51668d9d0e95607b61",
    "segmenter": "b881c338356bdf44aa0b45e0306f59d7b38252d04316da2455c40055aef656ec",
    "a3": "b19f0a5ece4f98fc08911f8781048011e14aeb84b80fde9dc1f4b9263322a02b",
    "m3_preds": "9dcc0576eb1c60d3e767ba9ca3c34a6a1b92b6b01946e377042dee1b1287b9e3",
    "b1": "f701e21261d690d9e3d7cb61b44ace3dc05660f82d4d977db2b92149f3e93251"}
OWNER = ACCOUNTS[ACCOUNT]


def dataset(handle):
    return Path(kagglehub.dataset_download(handle))


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def found(root, pattern):
    return next(Path(root).rglob(pattern))


DATA = dataset(f"{SHARED}/serpentguard-indian-snakes-clean")
SUBSET = dataset(f"{SHARED}/serpentguard-m3-subset") / "subset.csv"
MASKS = dataset(f"{SHARED}/serpentguard-sam-masks")
M5_INPUTS = dataset(f"{OWNER}/serpentguard-m5-inputs")
DETECTOR = found(M5_INPUTS, "M4_detector.pt")
SEGMENTER = found(M5_INPUTS, "segmenter/segmenter.pt")  # extend_predictions reads <folder>/segmenter.pt
A3_BEST = found(M5_INPUTS, "a3/best.pt")
M3_PREDS = found(dataset(f"{OWNER}/serpentguard-m4-frontend-masks"), front_end.PREDS_FILE).parent
B1_DIR = found(dataset(f"{OWNER}/serpentguard-m3-b1-predictions"), "predictions.npz").parent
M3_RUN = found(dataset(f"{OWNER}/serpentguard-m3-run-final"), VAL_OUTPUTS).parent
for name, path in (("detector", DETECTOR), ("segmenter", SEGMENTER), ("a3", A3_BEST),
                   ("m3_preds", M3_PREDS / front_end.PREDS_FILE), ("b1", B1_DIR / "predictions.npz")):
    assert sha256(path) == CHECKSUMS[name], f"{path} is not the {name} file this notebook expects"

ids = {"obs_id": str, "photo_id": str, "observer_id": str}
metadata = pd.read_csv(DATA / "metadata.csv", dtype=ids, low_memory=False)
classes = pd.read_csv(DATA / "classes.csv")
subset = pd.read_csv(SUBSET, dtype=ids)
taxa = class_tables(classes)
print(f"{len(metadata):,} photos of {len(classes)} species; every checksum matches")
PASSED.add("inputs")

### B1's saved predictions

As in notebook 21: B1's saved predictions must still give M3's recorded B1 scores before any GPU time.
The G3 headline macro-F1 is over the 62 species, on the photos of those species only.

In [ ]:
SLICE_FROM = "2022-01-01"
SLICE_PHOTOS, B1_SPECIES = 13_947, 62

b1 = load_outputs(B1_DIR / "predictions.npz")
b1_metrics = json.loads((B1_DIR / "final_metrics.json").read_text())
m3_comparison = json.loads((B1_DIR / "compare_b1_shared.json").read_text())
B1_CLASSES = np.array(sorted(b1_metrics["coverage"]["covered_class_ids"]))
assert len(B1_CLASSES) == B1_SPECIES == m3_comparison["shared_classes"]


def uploaded_2022(paths):
    created = pd.DataFrame({"path": paths}).merge(metadata[["path", "created_at"]], on="path",
                                                  how="left", validate="many_to_one")
    assert created["created_at"].notna().all(), "a photo has no upload date"
    return created["created_at"].astype(str).ge(SLICE_FROM).to_numpy()


def top1(y_true, pred):
    return float(np.mean(y_true == pred))


def macro_f1(y_true, pred):
    present = np.intersect1d(B1_CLASSES, y_true)
    return float(f1_score(y_true, pred, labels=present, average="macro", zero_division=0))


b1_pred = b1["probs"].argmax(axis=1)
keep = uploaded_2022(b1["paths"])
assert keep.sum() == SLICE_PHOTOS
shared_rows = keep & np.isin(b1["labels"], B1_CLASSES)
recorded = m3_comparison["upload_2022_slice"]["b1"]
assert shared_rows.sum() == recorded["photos"]
assert abs(top1(b1["labels"][shared_rows], b1_pred[shared_rows]) - recorded["top1"]) < 1e-9
PASSED.add("B1 check")
print(f"B1 on the 2022 slice: {shared_rows.sum():,} photos of its species, top-1 "
      f"{recorded['top1']:.4f} as recorded")

In [ ]:

SETTINGS = {  # name: (losses added to A3's, kd_weight, kd_temperature)
    "no kd": ([], None, None),
    "kd w1 T2 (the full run)": (["kd"], 1.0, 2.0),
    "kd w3 T4": (["kd"], 3.0, 4.0),
}
PILOT_EPOCHS = 2
SUBSET_TRAIN = data.cap_train(metadata[(metadata["split"] == "train") & metadata["path"].isin(subset["path"])])
calibration = data.calibration_set(metadata[metadata["split"] == "train"], subset)
unknown = data.unknown_split(metadata[metadata["split"] == "unknown_pool"])[0]
val_rows = metadata[metadata["split"] == "val"]
front_end_rows = pd.concat([SUBSET_TRAIN[["path"]], calibration[["path"]], val_rows[["path"]],
                            unknown[["path"]]]).drop_duplicates("path").reset_index(drop=True)
assert_no_test_paths(front_end_rows)
print(f"pilot: {len(SUBSET_TRAIN):,} subset training photos; the front end covers {len(front_end_rows):,}")

TEACHER_DIR = SCRATCH / "teacher"
TEACHER_RUN = TEACHER_DIR / "b1_subset.npz"
TEACHER_DIR.mkdir(parents=True, exist_ok=True)
labelled = SUBSET_TRAIN.merge(classes[["species", "class_id"]], on="species", how="left", validate="many_to_one")
labelled[labelled["class_id"].isin(B1_CLASSES)][["path"]].to_csv(TEACHER_DIR / "photos.csv", index=False)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "onnxruntime"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx==1.23.2", "onnxruntime-gpu==1.31.1"], check=True)
log = open(WORK / "b1_teacher.log", "w")
teacher_process = subprocess.Popen(
    [sys.executable, str(REPO / "scripts/b1_teacher.py"), str(DATA), str(TEACHER_DIR / "photos.csv"),
     str(B1_DIR / "predictions.npz"), str(SCRATCH / "downloads"), str(TEACHER_RUN)],
    stdout=log, stderr=subprocess.STDOUT,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "1" if torch.cuda.device_count() > 1 else "0"})

BANK = SCRATCH / "scenery_bank"
BANK_FROM = {"run_id": "scenery_bank", "front_end": "F1", "scenery_swap": 0.5,
             "mask_source": "predicted", "train_list": None, "scenery_bank": None}
BANK_SHA = "2eed027132bfacf956e85895b29131fa09f845c5b60bfcfa58272a687544fa5e"
a3 = data.run_config(yaml.safe_load((REPO / "results/m4/full/A3/config.yaml").read_text()))
bank = data.build_scenery_bank(data.make_frames({**a3, **BANK_FROM}, DATA, MASKS, SUBSET, M3_PREDS)["train"], BANK)
assert sha256(BANK / data.BANK_FILE) == BANK_SHA, "the bank differs from A3's"

EMPTY_BASE, PREDS = SCRATCH / "empty_base", SCRATCH / "m4_preds"
(EMPTY_BASE / front_end.MASKS_DIR).mkdir(parents=True, exist_ok=True)
(EMPTY_BASE / front_end.PREDS_FILE).write_text(front_end.PREDS_COLUMNS)
joined = front_end.extend_predictions(front_end_rows, EMPTY_BASE, DETECTOR, SEGMENTER, DATA, PREDS)
assert set(joined["path"]) == set(front_end_rows["path"]) and joined["path"].is_unique
assert_no_test_paths(joined)
print(f"M4 front end: {len(joined):,} rows, {joined['box'].ne('').mean():.1%} boxed, "
      f"{joined['mask_file'].ne('').mean():.1%} masked")


In [ ]:

while teacher_process.poll() is None:
    time.sleep(30)
assert teacher_process.returncode == 0, "B1 failed: read b1_teacher.log"
done = np.load(TEACHER_RUN)
val_known = np.isin(b1["labels"], B1_CLASSES)
TEACHER = TEACHER_DIR / "pilot_teacher.npz"
np.savez(TEACHER, paths=np.concatenate([done["paths"], b1["paths"][val_known]]).astype(str),
         probs=np.concatenate([done["probs"], b1["probs"][val_known]]).astype(np.float32))
print(f"teacher: {len(done['paths']):,} training photos, {val_known.sum():,} validation photos; "
      f"check {Path(f'{TEACHER_RUN}.check.json').read_text()}")


In [ ]:

def g3_slice(run_dir):
    ours = load_outputs(run_dir / VAL_OUTPUTS)
    labels = ours["labels"]
    b1_on = b1_pred[rows_for_b1(ours["paths"])]
    groups = photographer_groups(pd.DataFrame({"observer_id": ours["observer_id"], "path": ours["paths"]}))
    rows = uploaded_2022(ours["paths"]) & np.isin(labels, B1_CLASSES)
    assert rows.sum() == 11_931
    pred = ours["species_logits"].argmax(1)
    result = {}
    for metric, score in (("top1", top1), ("macro_f1", macro_f1)):
        diff, low, high = paired_difference_ci(labels[rows], pred[rows], b1_on[rows], groups[rows], score)
        result[metric] = {"diff": diff, "low": low, "high": high}
    shuffled = np.random.default_rng(0).permutation(pred)
    control = paired_difference_ci(labels[rows], shuffled[rows], b1_on[rows], groups[rows], top1)
    assert control[2] < -0.3, "the shuffled control must lose clearly"
    result["control_top1_shuffled"] = control
    return result


def rows_for_b1(paths):
    position = pd.Series(np.arange(len(b1["paths"])), index=b1["paths"]).reindex(paths)
    assert position.notna().all()
    return position.to_numpy(dtype=int)


table = {}
for name, (extra, weight, temperature) in SETTINGS.items():
    run_id = "PILOT_" + "".join(ch if ch.isalnum() else "_" for ch in name)
    cfg = {**a3, "run_id": run_id, "train_list": None, "epochs": PILOT_EPOCHS, "eval_every": 1, "lr": LR,
           "warmup_epochs": 1, "init_from": str(A3_BEST), "scenery_bank": str(BANK),
           "losses": [*a3["losses"], *extra]}
    if extra:
        cfg.update(teacher_probs=str(TEACHER), kd_weight=weight, kd_temperature=temperature)
    run_dir = WORK / "pilot" / run_id
    final = train(cfg, run_dir, DATA, MASKS, SUBSET, preds_root=PREDS)
    history = [json.loads(line) for line in (run_dir / "metrics.jsonl").read_text().splitlines()]
    table[name] = {"run": run_id, "train_photos": final["run"]["train_photos"],
                   "best_epoch": final["run"]["best_epoch"], "val_macro_f1": final["head"]["macro_f1"],
                   "val_top1": final["head"]["top1"], "history": history, "g3_slice": g3_slice(run_dir)}
    write_json(WORK / "pilot" / "pilot.json", table)
    g = table[name]["g3_slice"]
    print(f"{name}: val macro-F1 {final['head']['macro_f1']:.4f}, top-1 {final['head']['top1']:.4f}; G3 top-1 "
          f"{g['top1']['diff']:+.4f} ({g['top1']['low']:+.4f} to {g['top1']['high']:+.4f}), macro-F1 "
          f"{g['macro_f1']['diff']:+.4f} ({g['macro_f1']['low']:+.4f} to {g['macro_f1']['high']:+.4f})", flush=True)
